In [1]:
# =====================================================================
# onset_V4_input_types
# Same bigger CRNN, same data, same 3 seeds. Only the INPUT changes:
#   Mel-spectrogram (done before) | Spectrogram | MFCC | Raw audio
# Trained on VocalSet, tested on VocalSet + zero-shot on TONAS, SVNote1, CSD.
# =====================================================================


# %% Cell 1 — setup + clean VocalSet file list + split by singer
import torch, torchaudio, librosa
import torch.nn as nn
import torch.optim as optim
import pandas as pd, numpy as np
from pathlib import Path
import random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAMPLE_RATE = 44100
HOP_SECONDS = 0.010
HOP   = int(HOP_SECONDS * SAMPLE_RATE)   # 441 samples = 10 ms
N_FFT = 2048
LABELS = ["Rest", "Sound", "Onset", "Offset"]
label_to_id = {n: i for i, n in enumerate(LABELS)}

ROOT = Path("/kaggle/input/datasets/alirahmatpour/vocalset-onset")
ANNOT_CSV_DIR = ROOT / "annot_ext4_upload" / "annot_ext4_upload" / "annot"
AUDIO_DIR     = ROOT / "kaggle_upload" / "kaggle_upload" / "audio"
OUT_DIR       = Path("/kaggle/working")

COLS = ["Sequence","Start","End","Duration","Type","AvgF0","MedF0","MinF0","MaxF0",
        "StdF0","AvgF0_inSTD","EstMIDI","GTNote","GTFreq","GTMIDI","Lyric",
        "GTDurName","GTDur","IntPrev","IntNext","Extra"]

def read_annot(csv_path):
    a = pd.read_csv(csv_path, skiprows=17, header=None, names=COLS)
    return a[["Start","End","Type"]].dropna(subset=["Start","End","Type"])

# clean file list (same rule as the baseline)
csv_lookup = {p.stem: p for p in ANNOT_CSV_DIR.rglob("*.csv")}
pairs, dropped = [], 0
for wav in sorted(AUDIO_DIR.rglob("*.wav")):
    if wav.stem not in csv_lookup:
        continue
    wf2, sr2 = torchaudio.load(str(wav))
    dur = wf2.shape[1] / sr2
    ann = read_annot(csv_lookup[wav.stem])
    if not len(ann) or abs(ann["End"].max()/dur - 1.0) > 0.1:
        dropped += 1
        continue
    singer = wav.parts[wav.parts.index("audio") + 1]
    sex = "female" if singer.startswith("female") else "male"
    pairs.append({"name": wav.stem, "wav": wav, "csv": csv_lookup[wav.stem],
                  "singer": singer, "sex": sex})

# split by singer (fixed seed 42 -> same split as the baseline)
random.seed(42)
males   = sorted({p["singer"] for p in pairs if p["sex"]=="male"})
females = sorted({p["singer"] for p in pairs if p["sex"]=="female"})
random.shuffle(males); random.shuffle(females)
test_singers  = set(males[:1]  + females[:1])
val_singers   = set(males[1:3] + females[1:3])

def bucket(p):
    if p["singer"] in test_singers: return "test"
    if p["singer"] in val_singers:  return "val"
    return "train"

train_items = [p for p in pairs if bucket(p)=="train"]
val_items   = [p for p in pairs if bucket(p)=="val"]
test_items  = [p for p in pairs if bucket(p)=="test"]

print("device      :", device, "| GPU:", torch.cuda.is_available())
print("librosa     :", librosa.__version__)
print("clean files :", len(pairs), "| dropped:", dropped, "(expect 2537 / 147)")
print("train:", len(train_items), "| val:", len(val_items), "| test:", len(test_items),
      "(expect test 252)")
print("test singers:", sorted(test_singers), "(expect female4, male6)")

device      : cuda | GPU: True
librosa     : 0.11.0
clean files : 2537 | dropped: 147 (expect 2537 / 147)
train: 1719 | val: 566 | test: 252 (expect test 252)
test singers: ['female4', 'male6'] (expect female4, male6)


In [2]:
# %% Cell 2 — test sets for all 4 datasets (same code/splits as the 4x4 notebook)
def load_wave(path):
    """audio -> mono, 44.1 kHz"""
    wf, sr = torchaudio.load(str(path))
    w = wf.mean(0)
    if sr != SAMPLE_RATE:
        w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
    return w

def make_item(wave, on, test_onsets=None, clip_to_audio=False):
    """one test recording: wave + onset frames (one set per annotator)"""
    on = np.asarray(on, float)
    tests = test_onsets if test_onsets is not None else [on]
    nfr = 1 + len(wave) // HOP
    frames = []
    for t in tests:
        f = (np.asarray(t, float) / HOP_SECONDS).astype(int)
        if clip_to_audio:
            f = f[f < nfr]
        frames.append(set(f))
    return {"wave": wave, "frames": frames}

TEST = {}

# ---------- VocalSet test: singers female4 + male6 ----------
vs_test = []
for p in test_items:                       # test_items comes from cell 1
    wave = load_wave(p["wav"])
    ann = read_annot(p["csv"])
    snd = ann[ann["Type"] == "Sound"]
    x = make_item(wave, snd["Start"].values, clip_to_audio=True)
    x["sex"] = p["sex"]
    vs_test.append(x)
TEST["VocalSet"] = vs_test

# ---------- TONAS: singer split, Random(0) ----------
TONAS = ROOT / "tonas_upload"
tonas_pairs = [(w, w.parent / (w.stem + ".notes.Corrected")) for w in sorted(TONAS.rglob("*.wav"))
               if not w.name.startswith("._") and (w.parent / (w.stem + ".notes.Corrected")).exists()]

def tonas_singer(stem):
    return stem.split("_", 1)[1].split("-")[0] if "_" in stem else stem

t_items = []
for wav, notes in tonas_pairs:
    on = []
    for ln in notes.read_text(encoding="latin-1").strip().splitlines()[1:]:   # skip tuning line
        q = [v.strip() for v in ln.split(",")]
        if len(q) >= 2:
            on.append(float(q[0]))
    x = make_item(load_wave(wav), on)
    x["singer"] = tonas_singer(wav.stem)
    t_items.append(x)

rng = random.Random(0)
t_singers = sorted({x["singer"] for x in t_items}); rng.shuffle(t_singers)
t_test_singers, n_test = set(), 0
for sg in t_singers:
    if n_test >= len(t_items) // 3:
        break
    t_test_singers.add(sg); n_test += sum(x["singer"] == sg for x in t_items)
TEST["TONAS"] = [x for x in t_items if x["singer"] in t_test_singers]

# ---------- SVNote1: singer split, scored against each of the 3 annotators ----------
SV = ROOT / "svnote_upload"
ANNOTATORS = ["dooyong", "gwangseok", "sanghee"]
def sv_key(name):
    stem = name.split(".")[0]
    return "_".join(q for q in stem.split("_") if not q.isdigit())
sv_wavs = {sv_key(w.name): w for w in sorted((SV / "audio").rglob("*.wav")) if not w.name.startswith("._")}
sv_ann  = {a: {sv_key(c.name): c for c in (SV / "annot" / a).rglob("*.csv") if not c.name.startswith("._")}
           for a in ANNOTATORS}
sv_keys = sorted(set(sv_wavs).intersection(*[set(sv_ann[a]) for a in ANNOTATORS]))

s_items = []
for k in sv_keys:
    tabs = [np.loadtxt(sv_ann[a][k], delimiter=",", ndmin=2) for a in ANNOTATORS]
    x = make_item(load_wave(sv_wavs[k]), tabs[0][:, 0], test_onsets=[t[:, 0] for t in tabs])
    x["singer"] = k.split("_")[0]
    s_items.append(x)

rng = random.Random(0)
men   = sorted({x["singer"] for x in s_items if x["singer"].startswith("man")})
women = sorted({x["singer"] for x in s_items if x["singer"].startswith("woman")})
rng.shuffle(men); rng.shuffle(women)
s_test_singers = set(men[:2] + women[:1])
TEST["SVNote1"] = [x for x in s_items if x["singer"] in s_test_singers]

# ---------- CSD: song split, Random(0) per language ----------
CSD = ROOT / "csd_upload"
csd_index = pd.read_csv(CSD / "index.csv")
c_items = []
for r in csd_index.itertuples():
    lab = pd.read_csv(CSD / "annot" / f"{r.file}.csv")
    x = make_item(load_wave(CSD / "audio" / f"{r.file}.wav"), lab["start"].values)
    x["song"], x["lang"] = r.song, r.lang
    c_items.append(x)

rng = random.Random(0)
c_test_songs = set()
for lang in ["english", "korean"]:
    ss = sorted({x["song"] for x in c_items if x["lang"] == lang}); rng.shuffle(ss)
    c_test_songs |= set(ss[: len(ss) // 3])
TEST["CSD"] = [x for x in c_items if x["song"] in c_test_songs]

# ---------- checks ----------
print("dataset  | test files | test notes")
for name, items in TEST.items():
    print(f"{name:8s} | {len(items):10d} | {sum(len(x['frames'][0]) for x in items)}")
print("expected : VocalSet 252 / 2834 | TONAS 24 / 1066 | SVNote1 9 / 432 | CSD 34 / 5922")
print("SVNote1 test singers:", sorted(s_test_singers), "(expect man3, man5, woman1)")

dataset  | test files | test notes
VocalSet |        252 | 2834
TONAS    |         24 | 1066
SVNote1  |          9 | 432
CSD      |         34 | 5922
expected : VocalSet 252 / 2834 | TONAS 24 / 1066 | SVNote1 9 / 432 | CSD 34 / 5922
SVNote1 test singers: ['man3', 'man5', 'woman1'] (expect man3, man5, woman1)


In [3]:
# %% Cell 3 — the model with 4 input types (only the "front" changes)
N_MELS = 128
N_MFCC = 40
FREQ_OUT = N_MELS // 4          # 32: frequency size after the CNN for the Mel model

class OnsetCRNN_Input(nn.Module):
    """
    Same bigger CRNN as before. Only the input step changes:
      mel  : Mel-spectrogram, 128 bands, dB         (exactly the old model)
      spec : plain spectrogram, 1025 bins, dB
      mfcc : 40 MFCC coefficients
      raw  : the raw wave -> a learned filter layer (128 filters, one output every 10 ms)
    """
    def __init__(self, input_type="mel", dropout=0.3):
        super().__init__()
        self.input_type = input_type
        if input_type == "mel":
            self.mel   = torchaudio.transforms.MelSpectrogram(
                sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
            self.to_db = torchaudio.transforms.AmplitudeToDB()
        elif input_type == "spec":
            self.front = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=HOP)
            self.to_db = torchaudio.transforms.AmplitudeToDB()
        elif input_type == "mfcc":
            self.front = torchaudio.transforms.MFCC(
                sample_rate=SAMPLE_RATE, n_mfcc=N_MFCC,
                melkwargs={"n_fft": N_FFT, "hop_length": HOP, "n_mels": N_MELS})
        elif input_type == "raw":
            # 128 learned filters, window 2048 samples, one step = 441 samples (10 ms)
            self.front = nn.Conv1d(1, N_MELS, kernel_size=N_FFT, stride=HOP, padding=N_FFT // 2)
        else:
            raise ValueError(input_type)

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        n_in = {"mel": N_MELS, "spec": N_FFT // 2 + 1, "mfcc": N_MFCC, "raw": N_MELS}[input_type]
        self.freq_after_cnn = min(n_in // 4, FREQ_OUT)   # spec is squeezed to 32 so the GRU stays the same size
        self.squeeze = nn.AdaptiveAvgPool2d((self.freq_after_cnn, None))
        self.rnn = nn.GRU(128 * self.freq_after_cnn, 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head  = nn.Linear(256, 2)
        self.onset_head  = nn.Linear(256, 1)
        self.offset_head = nn.Linear(256, 1)

    def make_input(self, x):                      # x: (batch, 1, samples)
        if self.input_type == "mel":
            return self.to_db(self.mel(x))        # (B, 1, 128, T)
        if self.input_type == "spec":
            return self.to_db(self.front(x))      # (B, 1, 1025, T)
        if self.input_type == "mfcc":
            return self.front(x)                  # (B, 1, 40, T)
        z = self.front(x)                         # raw: (B, 128, T)
        return torch.log1p(z.abs()).unsqueeze(1)  # (B, 1, 128, T)

    def forward(self, x):
        s = self.cnn(self.make_input(x))
        s = self.squeeze(s)
        B, C, M, T = s.shape
        s, _ = self.rnn(s.reshape(B, C*M, T).transpose(1, 2))
        s = self.drop(s)
        return self.state_head(s), self.onset_head(s).squeeze(-1), self.offset_head(s).squeeze(-1)

INPUT_TYPES = ["mel", "spec", "mfcc", "raw"]

# ---------- checks: 3 seconds of audio must give 301 frames for every input ----------
dummy = torch.randn(1, 1, 3 * SAMPLE_RATE).to(device)
for it in INPUT_TYPES:
    m = OnsetCRNN_Input(it).to(device).eval()
    with torch.no_grad():
        st, on, off = m(dummy)
    n_par = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{it:5s} | input shape {tuple(m.make_input(dummy).shape)} | frames {on.shape[1]} (expect 301) | params {n_par:.2f} M")
    del m

# ---------- check: the old saved Mel weights load into the new model ----------
weight_files = sorted(Path("/kaggle/input").rglob("biggerCRNN_ext4_seed*.pt"))
print("\nsaved Mel weights:", [w.name for w in weight_files], "(expect 3)")
if weight_files:
    m = OnsetCRNN_Input("mel").to(device)
    m.load_state_dict(torch.load(weight_files[0], map_location=device))
    print("old Mel weights loaded OK -> no need to retrain Mel")
    del m
torch.cuda.empty_cache()

mel   | input shape (1, 1, 128, 301) | frames 301 (expect 301) | params 3.93 M
spec  | input shape (1, 1, 1025, 301) | frames 301 (expect 301) | params 3.93 M
mfcc  | input shape (1, 1, 40, 301) | frames 301 (expect 301) | params 1.77 M
raw   | input shape (1, 1, 128, 301) | frames 301 (expect 301) | params 4.19 M

saved Mel weights: ['biggerCRNN_ext4_seed0.pt', 'biggerCRNN_ext4_seed1.pt', 'biggerCRNN_ext4_seed2.pt'] (expect 3)
old Mel weights loaded OK -> no need to retrain Mel


In [4]:
# %% Cell 4 — training (same as the VocalSet baseline) + scoring (same as the 4x4 notebook)
import time

# ---------- training data: same as the baseline (soft labels, batch 8) ----------
SOFT_WIDTH = 2

def frame_labels(annot, n_frames):
    labs = np.full(n_frames, label_to_id["Rest"], dtype=np.int64)
    snd = annot[annot["Type"] == "Sound"]
    for _, r in snd.iterrows():
        labs[int(r["Start"]/HOP_SECONDS):int(r["End"]/HOP_SECONDS)] = label_to_id["Sound"]
    for _, r in snd.iterrows():
        off = int(r["End"]/HOP_SECONDS)
        if off < n_frames: labs[off] = label_to_id["Offset"]
    for _, r in snd.iterrows():
        on = int(r["Start"]/HOP_SECONDS)
        if on < n_frames: labs[on] = label_to_id["Onset"]
    return labs

def soft_targets(annot, n_frames):
    on_t, off_t = np.zeros(n_frames, np.float32), np.zeros(n_frames, np.float32)
    def bump(arr, c):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH + 1):
            j = c + d
            if 0 <= j < n_frames:
                arr[j] = max(arr[j], np.exp(-(d*d) / (2 * (SOFT_WIDTH/2)**2)))
    for _, r in annot[annot["Type"] == "Sound"].iterrows():
        bump(on_t, int(r["Start"]/HOP_SECONDS)); bump(off_t, int(r["End"]/HOP_SECONDS))
    return on_t, off_t

def state_from_frames(h):
    return np.where(h == label_to_id["Rest"], 0, 1).astype(np.int64)

class SeqDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p = self.items[i]
        wave = torchaudio.load(str(p["wav"]))[0][0]
        nfr  = 1 + wave.shape[0] // HOP
        ann  = read_annot(p["csv"])
        on, off = soft_targets(ann, nfr)
        return wave, torch.tensor(state_from_frames(frame_labels(ann, nfr))), \
               torch.tensor(on), torch.tensor(off), nfr

def collate(batch):
    waves, states, ons, offs, lens = zip(*batch)
    X   = pad_sequence(list(waves),  batch_first=True).unsqueeze(1)
    S   = pad_sequence(list(states), batch_first=True, padding_value=-100)
    On  = pad_sequence(list(ons),    batch_first=True, padding_value=0.0)
    Off = pad_sequence(list(offs),   batch_first=True, padding_value=0.0)
    mask = torch.zeros(len(batch), S.shape[1], dtype=torch.bool)
    for i, L in enumerate(lens): mask[i, :L] = True
    return X, S, On, Off, mask

BATCH = 8
val_loader = DataLoader(SeqDataset(val_items), batch_size=BATCH, shuffle=False,
                        collate_fn=collate, num_workers=2)
def make_train_loader(seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(SeqDataset(train_items), batch_size=BATCH, shuffle=True,
                      collate_fn=collate, num_workers=2, generator=g)

# ---------- loss: same as the baseline ----------
random.seed(42)
sc = np.zeros(2)
for p in random.sample(train_items, min(300, len(train_items))):
    nfr = 1 + torchaudio.load(str(p["wav"]))[0].shape[1] // HOP
    s = state_from_frames(frame_labels(read_annot(p["csv"]), nfr))
    sc[0] += (s == 0).sum(); sc[1] += (s == 1).sum()
state_w = torch.tensor(sc.sum() / (2 * np.maximum(sc, 1)), dtype=torch.float32).to(device)
state_loss_fn = nn.CrossEntropyLoss(weight=state_w, ignore_index=-100)
soft_loss_fn  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0).to(device))

def combined_loss(st, on, off, S, On, Off, mask):
    F = min(st.shape[1], S.shape[1]); m = mask[:, :F]
    return (state_loss_fn(st[:, :F].reshape(-1, 2), S[:, :F].reshape(-1))
            + soft_loss_fn(on[:, :F][m], On[:, :F][m]) + soft_loss_fn(off[:, :F][m], Off[:, :F][m]))

# ---------- train one model: same settings as the baseline ----------
EPOCHS, PATIENCE, MAX_FRAMES = 20, 4, 1500

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def train_one(input_type, seed):
    set_seed(seed)
    model = OnsetCRNN_Input(input_type).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = make_train_loader(seed)
    w_path = OUT_DIR / f"input_{input_type}_seed{seed}.pt"

    def run_epoch(loader, train):
        model.train() if train else model.eval()
        tot, n = 0.0, 0
        with torch.set_grad_enabled(train):
            for X, S, On, Off, mask in loader:
                X = X[:, :, :MAX_FRAMES * HOP].to(device)
                S, On, Off = S[:, :MAX_FRAMES].to(device), On[:, :MAX_FRAMES].to(device), Off[:, :MAX_FRAMES].to(device)
                mask = mask[:, :MAX_FRAMES].to(device)
                st, on, off = model(X)
                loss = combined_loss(st, on, off, S, On, Off, mask)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item() * X.size(0); n += X.size(0)
        torch.cuda.empty_cache()
        return tot / n

    best_val, best_epoch, wait, t0 = float("inf"), 0, 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        tr, va = run_epoch(train_loader, True), run_epoch(val_loader, False)
        flag = ""
        if va < best_val:
            best_val, best_epoch, wait = va, epoch, 0
            torch.save(model.state_dict(), w_path); flag = "  <-- best (saved)"
        else:
            wait += 1
        print(f"[{input_type} seed {seed}] epoch {epoch:2d} | train {tr:.3f} | val {va:.3f} | "
              f"{(time.time()-t0)/60:.0f} min{flag}")
        if wait >= PATIENCE:
            print(f"[{input_type} seed {seed}] early stop"); break
    model.load_state_dict(torch.load(w_path, map_location=device))
    return model, best_epoch

# ---------- scoring: same as the 4x4 notebook (threshold 0.8, ±20 and ±50 ms) ----------
def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i - peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def f1_sets(pred_list, true_list, tol):
    tp = fp = fn = 0
    for pred, t_idx in zip(pred_list, true_list):
        matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi - ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(set(pred) - matched)
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

def evaluate_on(model, test_items, thr=0.8):
    model.eval()
    with torch.no_grad():
        peaks = [pick_peaks(torch.sigmoid(model(x["wave"].view(1, 1, -1).to(device))[1])[0].cpu().numpy(), thr)
                 for x in test_items]
    n_ann = len(test_items[0]["frames"])
    return {f"F1_{n}": np.mean([f1_sets(peaks, [x["frames"][j] for x in test_items], tol)
                                for j in range(n_ann)])
            for n, tol in [("20", 2), ("50", 5)]}

def test_all(model, input_type, seed, rows):
    for ts in ["VocalSet", "TONAS", "SVNote1", "CSD"]:
        r = evaluate_on(model, TEST[ts])
        rows.append({"input": input_type, "tested_on": ts, "seed": seed, **r})
        print(f"  {input_type:5s} seed {seed} -> {ts:8s} | ±20 {r['F1_20']:.3f} | ±50 {r['F1_50']:.3f}")

print("train:", len(train_items), "| val:", len(val_items), "| state weights:", state_w.cpu().numpy().round(3))
print("tools ready")

train: 1719 | val: 566 | state weights: [3.734 0.577]
tools ready


In [5]:
# %% Cell 4b — free GPU memory + train in small groups (same 8 files per update)
import gc, sys
sys.last_traceback = sys.last_value = sys.last_type = None   # drop the error's memory
gc.collect(); torch.cuda.empty_cache()
print("GPU memory in use:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB (should be small)")

MICRO = {"mel": 8, "spec": 2, "mfcc": 8, "raw": 8}   # files per small group; update still uses all 8

def train_one(input_type, seed):
    set_seed(seed)
    model = OnsetCRNN_Input(input_type).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = make_train_loader(seed)
    w_path = OUT_DIR / f"input_{input_type}_seed{seed}.pt"
    mb = MICRO[input_type]

    def run_epoch(loader, train):
        model.train() if train else model.eval()
        tot, n = 0.0, 0
        with torch.set_grad_enabled(train):
            for X, S, On, Off, mask in loader:
                X = X[:, :, :MAX_FRAMES * HOP].to(device)
                S, On, Off = S[:, :MAX_FRAMES].to(device), On[:, :MAX_FRAMES].to(device), Off[:, :MAX_FRAMES].to(device)
                mask = mask[:, :MAX_FRAMES].to(device)
                B = X.size(0)
                if train: opt.zero_grad()
                batch_loss = 0.0
                for a in range(0, B, mb):                      # small groups
                    b = min(B, a + mb)
                    st, on, off = model(X[a:b])
                    loss = combined_loss(st, on, off, S[a:b], On[a:b], Off[a:b], mask[a:b]) * (b - a) / B
                    if train: loss.backward()                   # gradients add up over the groups
                    batch_loss += loss.item()
                    del st, on, off, loss
                if train: opt.step()                            # one update per 8 files, as before
                tot += batch_loss * B; n += B
        torch.cuda.empty_cache()
        return tot / n

    best_val, best_epoch, wait, t0 = float("inf"), 0, 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        tr, va = run_epoch(train_loader, True), run_epoch(val_loader, False)
        flag = ""
        if va < best_val:
            best_val, best_epoch, wait = va, epoch, 0
            torch.save(model.state_dict(), w_path); flag = "  <-- best (saved)"
        else:
            wait += 1
        print(f"[{input_type} seed {seed}] epoch {epoch:2d} | train {tr:.3f} | val {va:.3f} | "
              f"{(time.time()-t0)/60:.0f} min{flag}")
        if wait >= PATIENCE:
            print(f"[{input_type} seed {seed}] early stop"); break
    model.load_state_dict(torch.load(w_path, map_location=device))
    return model, best_epoch

print("low-memory training ready")

GPU memory in use: 0.01 GB (should be small)
low-memory training ready


In [6]:
# %% Cell 5 — Mel: score the 3 saved models (no training) -> must match the old numbers
rows = []
for k, w in enumerate(weight_files):
    model = OnsetCRNN_Input("mel").to(device)
    model.load_state_dict(torch.load(w, map_location=device))
    test_all(model, "mel", k, rows)
    del model; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "input_types_results.csv", index=False)

print("\n=== Mel, mean of 3 seeds (±50 ms) ===")
print(res.groupby("tested_on")["F1_50"].mean().reindex(["VocalSet", "TONAS", "SVNote1", "CSD"]).round(3).to_string())
print("expect: VocalSet 0.926 | TONAS 0.430 | SVNote1 0.697 | CSD 0.770")

  mel   seed 0 -> VocalSet | ±20 0.837 | ±50 0.923
  mel   seed 0 -> TONAS    | ±20 0.139 | ±50 0.411
  mel   seed 0 -> SVNote1  | ±20 0.402 | ±50 0.633
  mel   seed 0 -> CSD      | ±20 0.500 | ±50 0.755
  mel   seed 1 -> VocalSet | ±20 0.847 | ±50 0.933
  mel   seed 1 -> TONAS    | ±20 0.130 | ±50 0.394
  mel   seed 1 -> SVNote1  | ±20 0.418 | ±50 0.743
  mel   seed 1 -> CSD      | ±20 0.538 | ±50 0.764
  mel   seed 2 -> VocalSet | ±20 0.819 | ±50 0.921
  mel   seed 2 -> TONAS    | ±20 0.201 | ±50 0.486
  mel   seed 2 -> SVNote1  | ±20 0.444 | ±50 0.713
  mel   seed 2 -> CSD      | ±20 0.570 | ±50 0.792

=== Mel, mean of 3 seeds (±50 ms) ===
tested_on
VocalSet    0.926
TONAS       0.430
SVNote1     0.697
CSD         0.770
expect: VocalSet 0.926 | TONAS 0.430 | SVNote1 0.697 | CSD 0.770


In [7]:
# %% Cell 6 — train MFCC and Raw (3 seeds each), test on all 4 datasets
for INPUT in ["mfcc", "raw"]:
    for seed in [0, 1, 2]:
        model, best_epoch = train_one(INPUT, seed)
        print(f"[{INPUT} seed {seed}] best epoch {best_epoch}")
        test_all(model, INPUT, seed, rows)
        pd.DataFrame(rows).to_csv(OUT_DIR / "input_mfcc_raw_results.csv", index=False)   # save after every seed
        del model; torch.cuda.empty_cache()
        print()

# ---------- summary: mean ± std of 3 seeds ----------
res = pd.DataFrame(rows)
order = ["VocalSet", "TONAS", "SVNote1", "CSD"]
for metric in ["F1_50", "F1_20"]:
    print(f"\n=== {metric.replace('F1_', '±')} ms — mean ± std of 3 seeds ===")
    m = res.groupby(["input", "tested_on"])[metric].mean().unstack()[order]
    s = res.groupby(["input", "tested_on"])[metric].std(ddof=1).unstack()[order]
    for inp in [i for i in ["mel", "mfcc", "raw"] if i in m.index]:
        print(f"{inp:5s} | " + " | ".join(f"{c} {m.loc[inp, c]:.3f}±{s.loc[inp, c]:.3f}" for c in order))

[mfcc seed 0] epoch  1 | train 1.550 | val 1.237 | 1 min  <-- best (saved)
[mfcc seed 0] epoch  2 | train 1.006 | val 1.102 | 1 min  <-- best (saved)
[mfcc seed 0] epoch  3 | train 0.912 | val 1.046 | 2 min  <-- best (saved)
[mfcc seed 0] epoch  4 | train 0.826 | val 0.981 | 3 min  <-- best (saved)
[mfcc seed 0] epoch  5 | train 0.774 | val 0.981 | 3 min  <-- best (saved)
[mfcc seed 0] epoch  6 | train 0.754 | val 0.977 | 4 min  <-- best (saved)
[mfcc seed 0] epoch  7 | train 0.724 | val 0.961 | 5 min  <-- best (saved)
[mfcc seed 0] epoch  8 | train 0.706 | val 0.957 | 5 min  <-- best (saved)
[mfcc seed 0] epoch  9 | train 0.687 | val 0.944 | 6 min  <-- best (saved)
[mfcc seed 0] epoch 10 | train 0.662 | val 0.970 | 7 min
[mfcc seed 0] epoch 11 | train 0.649 | val 0.971 | 8 min
[mfcc seed 0] epoch 12 | train 0.626 | val 0.961 | 8 min
[mfcc seed 0] epoch 13 | train 0.600 | val 0.978 | 9 min
[mfcc seed 0] early stop
[mfcc seed 0] best epoch 9
  mfcc  seed 0 -> VocalSet | ±20 0.811 | ±50 